## Load GNSS data

Load collected UART serial output from the Heltec ESP32 + L76K equipment.


In [1]:
import re
from pathlib import Path

import pandas as pd

# Match only lines that carry an actual GNSS fix (position data present).
# "waiting for fix..." lines carry no position and are dropped.
FIX_RE = re.compile(
    r"^(?P<time>\d{2}:\d{2}:\d{2}\.\d{3}) GNSS fix: "
    r"lat=(?P<lat>-?\d+\.\d+), lon=(?P<lon>-?\d+\.\d+), "
    r"alt=(?P<alt>-?\d+\.\d+)m, sats=(?P<sats>\d+), "
    r"hdop=(?P<hdop>\d+\.\d+), sog=(?P<sog>\d+\.\d+)kn, "
    r"cog=(?P<cog>\d+\.\d+), (?P<quality>OK|LOW-QUALITY)$"
)

input_dir = Path("input_data")
rows = []
for path in sorted(input_dir.glob("*.txt")):
    with path.open() as f:
        for line in f:
            match = FIX_RE.match(line.strip())
            if match:
                rows.append(match.groupdict())

df = pd.DataFrame(rows)

numeric_cols = ["lat", "lon", "alt", "sats", "hdop", "sog", "cog"]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)
df["time"] = pd.to_datetime(df["time"], format="%H:%M:%S.%f").dt.time

# df.head()

## Convert to local meters

Convert lat/lon readings into a meters-based deltas and offset using an equirectangular approximation method.

In [2]:
import numpy as np

EARTH_RADIUS_M = 6371000  # mean Earth radius (spherical approximation)

# Equirectangular approximation: fine for the short distances covered in a
# single run. Origin is the run's first fix, so lat/lon are replaced with
# a plain meters-north / meters-east offset from the start — no absolute
# GPS coordinates need to leave this notebook.
lat0_rad = np.radians(df["lat"].iloc[0])
lon0_rad = np.radians(df["lon"].iloc[0])
lat_rad = np.radians(df["lat"])
lon_rad = np.radians(df["lon"])

df["x_m"] = (lon_rad - lon0_rad) * np.cos(lat0_rad) * EARTH_RADIUS_M
df["y_m"] = (lat_rad - lat0_rad) * EARTH_RADIUS_M

df = df.drop(columns=["lat", "lon"])

df["dx_m"] = df["x_m"].diff()
df["dy_m"] = df["y_m"].diff()

df.head()

,time,alt,sats,hdop,sog,cog,quality,x_m,y_m,dx_m,dy_m
0,21:50:07.228000,58.4,4,25.5,0.00,0.0,LOW-QUALITY,0.000000,0.000000,NaN,NaN
1,21:50:08.222000,58.3,4,1.5,0.01,0.0,OK,0.000000,0.111195,0.000000,0.111195
2,21:50:09.225000,58.3,4,1.5,0.05,0.0,OK,-0.096071,0.111195,-0.096071,0.000000
3,21:50:10.224000,58.2,4,1.5,0.24,0.0,OK,-0.288212,0.111195,-0.192141,0.000000
4,21:50:11.227000,58.3,4,1.5,0.31,0.0,OK,-0.480353,0.111195,-0.192141,0.000000
